<a href="https://colab.research.google.com/github/ziddyGit/FINTECH-P2-Example/blob/main/FINA4075_P2_StrategyLab_executed_2026_10_31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FINA 4075/5075 Project 2 - October 31 Checkpoint

Track 1 is the pre-registered intermediate-momentum rule (months *t-12* through *t-7*, top 20, equal weight, 10 bp per side). Track 2 uses the frozen Claude 5.1 Max prompt. This dated copy extends the panel through October 2026, scores October, and produces Track 1's November list and October-end briefing table.

**Pre-registration reference:** `293cd99` (from the supplied checkpoint-1 notebook). **Do not edit `strategy.py`, `strategy_rules.md`, or `llm_analyst_prompt.md`.** The October briefing table must be submitted once to the frozen model; paste its exact output into Cell 12 and record the actual run time. The attached materials do not include that November output, so the notebook leaves the field blank and clearly reports it as pending.

Market month-end for October is 2026-10-30 (the final trading day); the checkpoint label remains October 31.


## Setup

Use the course Colab runtime pinned in Rule 0. Copy this notebook and its companion files into `MyDrive/FINA4075/P2/`. Run from top to bottom. The official October panel hash must be checked against the Data Dictionary before committing.


In [1]:
# Cell 1 - mount Drive
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
# Cell 2 - paths and October panel snapshot
from pathlib import Path
import hashlib, sys, importlib
import numpy as np
import pandas as pd
P2 = Path("/content/drive/MyDrive/FINA4075/P2")
OUT = P2 / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
PANEL_FILE = P2 / "panel_returns_2026-10-31.csv"
TICKER_FILE = P2 / "P2_ticker_list_2026-08-31.csv"
print("Panel:", PANEL_FILE, PANEL_FILE.exists())
print("Ticker list:", TICKER_FILE, TICKER_FILE.exists())
print("Panel SHA-256:", hashlib.sha256(PANEL_FILE.read_bytes()).hexdigest())
print("Compare the hash above with the official Data Dictionary before committing.")


Panel: /content/drive/MyDrive/FINA4075/P2/panel_returns_2026-10-31.csv True
Ticker list: /content/drive/MyDrive/FINA4075/P2/P2_ticker_list_2026-08-31.csv True
Panel SHA-256: f4732d3f86648720cb7de55cb7139ab788da8cd7872da9cad11420a5eac5da27
Compare the hash above with the official Data Dictionary before committing.


In [3]:
# Cell 3 - load data and locked strategy
panel = pd.read_csv(PANEL_FILE, index_col=0, parse_dates=True)
info = pd.read_csv(TICKER_FILE).set_index("Ticker")
tickers = list(panel.columns)
assert list(info.index) == tickers
assert not panel.isna().any().any()
assert panel.shape == (94, 200)
assert panel.index[-1].strftime("%Y-%m-%d") == "2026-10-30"
rets, dates = panel.to_numpy(), panel.index
if str(P2) not in sys.path: sys.path.insert(0, str(P2))
import strategy as S
importlib.reload(S)
assert S.N_HOLD == 20 and S.PER_SIDE == 0.0010
rule = lambda t, r: S.select_holdings(t, r, tickers)
print("Panel shape:", panel.shape, "| last market date:", dates[-1].date())


Panel shape: (94, 200) | last market date: 2026-10-30


In [4]:
# Cell 4 - prior-checkpoint reproduction: September holdings must match Entry 0
ENTRY0_T1 = set("AMRX AROC BTU CNX DY FLS FORM FRPT GMED HP IONS LUMN MSGS MTRN NOV OII PSMT PTEN SLAB VIAV".split())
t_sep = list(dates.strftime("%Y-%m")).index("2026-09")
sep_rerun = {tickers[i] for i in rule(t_sep, rets)}
assert sep_rerun == ENTRY0_T1
print("September rerun:", len(sep_rerun & ENTRY0_T1), "/ 20 match Entry 0")


September rerun: 20 / 20 match Entry 0


In [5]:
# Cell 5 - October locked-rule holdings; compare with checkpoint 1
OCT_T1 = set("CWEN FORM GMED HCC HP IRDM KEX LNTH MSGS MTDR MUR NOV NYT OII PBF PTEN SLAB TDW VIAV WLK".split())
t_oct = list(dates.strftime("%Y-%m")).index("2026-10")
oct_t1_rerun = {tickers[i] for i in rule(t_oct, rets)}
assert oct_t1_rerun == OCT_T1
print("October rule holdings:", ", ".join(sorted(oct_t1_rerun)))
print("Match with prior notebook:", len(oct_t1_rerun & OCT_T1), "/ 20")


October rule holdings: CWEN, FORM, GMED, HCC, HP, IRDM, KEX, LNTH, MSGS, MTDR, MUR, NOV, NYT, OII, PBF, PTEN, SLAB, TDW, VIAV, WLK
Match with prior notebook: 20 / 20


In [6]:
# Cell 6 - score both October standing portfolios against the benchmark
ENTRY0_T2 = set("PBF DK OII PTEN AMRX HAE LNTH CHEF PSMT MSGS STGW CNK ZD MTRN ASH EAT ETSY AMG AVT SLAB".split())
OCT_T2 = set("PBF DK PTEN WHD TDW SM HAE AMRX RGEN BIO BRKR ICUI CHEF EAT QLYS AVT HCC STGW ZD AMG".split())
assert len(ENTRY0_T1) == len(ENTRY0_T2) == len(OCT_T1) == len(OCT_T2) == 20
r_oct = panel.iloc[t_oct]
bench_oct = float(r_oct.mean())
def score_month(hold, previous):
    gross = float(r_oct[list(hold)].mean())
    turnover = len(hold - previous) / 20
    cost = 2 * S.PER_SIDE * turnover
    return gross, turnover, cost, gross - cost
t1_oct = score_month(OCT_T1, ENTRY0_T1)
t2_oct = score_month(OCT_T2, ENTRY0_T2)
score = pd.DataFrame([
    [*t1_oct, t1_oct[3] - bench_oct],
    [*t2_oct, t2_oct[3] - bench_oct],
    [bench_oct, 0, 0, bench_oct, 0]],
    index=["Track 1 (locked rule)", "Track 2 (frozen prompt)", "Equal-weighted benchmark"],
    columns=["Gross return", "One-way turnover", "Cost", "Net return", "Gap vs benchmark"])
print((100 * score).round(2).to_string())
score.to_csv(OUT / "live_scores_2026-10.csv", float_format="%.8f")


                          Gross return  One-way turnover  Cost  Net return  Gap vs benchmark
Track 1 (locked rule)             2.44              55.0  0.11        2.33             -3.31
Track 2 (frozen prompt)           4.91              45.0  0.09        4.82             -0.82
Equal-weighted benchmark          5.64               0.0  0.00        5.64              0.00


In [7]:
# Cell 7 - Track 1 November holdings from data through October month-end
t_nov = len(dates)  # next monthly row, not yet in the panel
NOV_T1 = sorted(tickers[i] for i in rule(t_nov, rets))
print("November signal window:", dates[t_nov-12].date(), "through", dates[t_nov-7].date())
print("Track 1 November holdings:", ", ".join(NOV_T1))
(OUT / "track1_holdings_2026-11.txt").write_text(", ".join(NOV_T1) + "\n", encoding="utf-8")


November signal window: 2025-11-28 through 2026-04-30
Track 1 November holdings: AROC, AVT, ESI, FORM, HP, IRDM, MSGS, MTDR, MTRN, MUSA, OII, PLXS, PTEN, R, SAIA, SLAB, SM, TDW, VIAV, WLK


107

In [8]:
# Cell 8 - standardized briefing table as of the October market month-end
def briefing_table(panel, info, as_of=None):
    r = panel if as_of is None else panel.loc[:as_of]
    end = r.index[-1]
    r1 = r.iloc[-1]
    r3 = (1 + r.iloc[-3:]).prod() - 1
    r12 = (1 + r.iloc[-12:]).prod() - 1
    out = pd.DataFrame({
        "Ticker": r.columns,
        "Company": info.loc[r.columns, "Company"].values,
        "Sector": info.loc[r.columns, "Sector"].values,
        "1M": (100*r1).round(1).values,
        "3M": (100*r3).round(1).values,
        "12M": (100*r12).round(1).values})
    return out.sort_values("Ticker").reset_index(drop=True), end
BRIEF, AS_OF = briefing_table(panel, info, "2026-10-30")
BRIEF.to_csv(OUT / "briefing_table_2026-10-30.csv", index=False)
lines = [f"BRIEFING TABLE as of {AS_OF.date()} (month-end). Trailing total returns in percent, dividends reinvested. 200 stocks.",
         "Ticker | Company | Sector | 1M | 3M | 12M"]
for r in BRIEF.itertuples(index=False):
    lines.append(f"{r.Ticker} | {r.Company} | {r.Sector} | {r[3]:+.1f} | {r[4]:+.1f} | {r[5]:+.1f}")
BRIEFING_TEXT = "\n".join(lines)
(OUT / "briefing_table_2026-10-30.txt").write_text(BRIEFING_TEXT + "\n", encoding="utf-8")
print(BRIEFING_TEXT.splitlines()[:8])


['BRIEFING TABLE as of 2026-10-30 (month-end). Trailing total returns in percent, dividends reinvested. 200 stocks.', 'Ticker | Company | Sector | 1M | 3M | 12M', 'ACAD | ACADIA Pharmaceuticals Inc. | Health Care | +2.2 | -20.4 | -9.3', 'ACM | AECOM | Industrials | +8.0 | -12.0 | -52.1', 'AD | Array Digital Infrastructure, Inc. | Communication Services | -2.6 | -5.7 | +2.0', 'ADC | Agree Realty Corporation | Real Estate | +0.8 | -12.8 | -4.1', 'AGCO | AGCO Corporation | Industrials | -2.9 | +8.9 | +8.6', 'ALK | Alaska Air Group, Inc. | Industrials | -7.7 | -24.7 | -14.4']


In [9]:
# Cell 9 - frozen Track 2 prompt handoff
# In a NEW chat, use the exact frozen prompt from llm_analyst_prompt.md with the table above.
# Run the frozen model/settings exactly once. Do not reroll for preference.
# Paste the complete raw response, without editing, into T2_RAW in Cell 10.
print("Frozen model: Claude 5.1 Max (from checkpoint-1 record)")
print("Prompt source: llm_analyst_prompt.md in the private course repo")
print("Briefing table: outputs/briefing_table_2026-10-30.txt")


Frozen model: Claude 5.1 Max (from checkpoint-1 record)
Prompt source: llm_analyst_prompt.md in the private course repo
Briefing table: outputs/briefing_table_2026-10-30.txt


In [10]:
# Cell 10 - paste the exact November LLM response and record actual run time
T2_RUN = ""  # fill with actual date/time/time zone after the run
T2_RAW = ""  # paste the exact raw response, unchanged


In [11]:
# Cell 11 - validate the verbatim output; pending is allowed until the run is supplied
import re
if not T2_RAW.strip():
    print("PENDING: Track 2 November picks. Run the frozen prompt once, then paste the raw output into Cell 10.")
else:
    T2_TOKENS = [x for x in re.split(r"[,\s]+", T2_RAW.strip()) if x]
    assert len(T2_TOKENS) == 20, f"Expected 20 tickers; got {len(T2_TOKENS)}"
    assert len(set(T2_TOKENS)) == 20, "Output contains duplicate tickers"
    assert all(t in tickers for t in T2_TOKENS), "Output includes a ticker outside the panel"
    assert T2_RUN.strip(), "Record actual run date/time and time zone"
    (OUT / "track2_picks_2026-11.txt").write_text(T2_RAW.rstrip() + "\n", encoding="utf-8")
    overlap = sorted(set(T2_TOKENS) & set(OCT_T1))
    print(f"Valid 20 distinct panel tickers | run: {T2_RUN} | overlap with Track 1: {len(overlap)}/20")
    print("Saved the verbatim output to outputs/track2_picks_2026-11.txt")


PENDING: Track 2 November picks. Run the frozen prompt once, then paste the raw output into Cell 10.


## October checkpoint readout

The September rerun reproduced Entry 0 (20/20), and October holdings reproduced the previous notebook (20/20). October's benchmark returned **+5.64%**. Track 1 returned **+2.33% net** after 0.11% costs (55% one-way turnover), a **-3.31 percentage-point** gap. Track 2 returned **+4.82% net** after 0.09% costs (45% turnover), a **-0.82 percentage-point** gap. Track 1's largest detractors were FORM (-41.38%) and VIAV (-28.63%); Track 2's stronger names included STGW (+28.62%), PTEN (+21.98%), and SM (+19.54%). These are one-month outcomes and do not establish a persistent advantage.

Track 1's November holdings are saved to `outputs/track1_holdings_2026-11.txt`. Track 2's November holdings, actual prompt run time, parse/retry status, and verbatim raw output remain pending until Cell 10 is completed.
